# External Framework Validation — FEST

**Clean, linear pipeline.** Run top to bottom (Kernel -> Restart & Run All).

This notebook evaluates the CTGAN and CTAB-GAN+ synthetic datasets with **FEST**
(Niu et al., 2025), a unified framework that combines distance-based privacy
metrics, statistical similarity metrics, and machine-learning utility metrics in
a single interface. FEST is the fifth and final external framework applied to
the main dataset.

Following the agreed full scope, this notebook computes, for each synthetic
model:

* **Privacy (distance-based):** DCR, NNDR, and Nearest-Neighbour Adversarial
  Accuracy (NNAA / AdversarialAccuracy). DCR and NNDR triangulate directly with
  the manual DCR/NNDR analysis; NNAA is a new metric not computed
  by any of the other four frameworks (the acronym NNAA is already reserved in
  the thesis preamble).
* **Similarity / utility:** Wasserstein distance, KS similarity, correlation
  (Pearson and Spearman), mutual information, Jensen-Shannon similarity, and
  basic statistics. These triangulate with SynthEval, Synthcity and SDMetrics.

The framework's own API is used exactly as documented in its repository examples
(`examples/privacy_distance.py`, `examples/utility.py`): each metric is a
`Calculator` built from the original and synthetic dataframes, collected in a
`PrivacyMetricManager` or `UtilityMetricManager` and run with `evaluate_all()`.

To stay consistent with the other frameworks, the same files are used:
`real_minority_train.csv` as the real reference and
`synthetic_ctgan_final.csv` / `synthetic_ctab_final.csv` as the synthetic sets.

**Requirements:** environment `tese_fest` (Python 3.10) with
`privacy_utility_framework==0.0.11` installed from the FEST repository.


## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# >>> Adjust this path if necessary <<<
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)


Data folder: data


## 1. Canonical column definitions

Identical to the other framework notebooks: 23 features plus the target. FEST's
distance metrics operate on the whole record and preprocess internally
(`preprocess=True`), so the full record including `def_pay` is passed.


In [2]:
TARGET   = "def_pay"

FEATURES = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
            "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
            "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
            "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

ALL_COLS = FEATURES + [TARGET]

MONETARY = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] \
                         + [f"PAY_AMT{i}"  for i in range(1, 7)]

assert len(FEATURES) == 23
print("Working schema:", len(ALL_COLS), "columns")


Working schema: 24 columns


## 2. Helper: align a dataframe to the canonical schema

Keeps the 23 features plus target, enforces order, and casts monetary columns to
`float64`. FEST handles categorical encoding internally when `preprocess=True`,
so no manual categorical casting is needed here; the columns just need to match
across the real and synthetic frames.


In [3]:
def align(df, name):
    d = df.copy()
    assert TARGET in d.columns, f"{name}: '{TARGET}' column is required"
    d = d[ALL_COLS]
    d[MONETARY] = d[MONETARY].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d


## 3. Load the real and synthetic datasets

In [4]:
real      = align(pd.read_csv(DATA_DIR / "real_minority_train.csv"),  "real (min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv"),  "syn ctabgan+")

for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(real.columns)
print("\nColumns aligned.")


real (min)       -> (4645, 24)
syn ctgan        -> (11710, 24)
syn ctabgan+     -> (11710, 24)

Columns aligned.


## 4. Privacy metrics (distance-based)

Uses `PrivacyMetricManager` with the three distance-based calculators, exactly
as in `examples/privacy_distance.py`. `preprocess=True` lets FEST transform and
scale the data internally. The NNAA calculator uses `backend="auto"`, which
selects an appropriate nearest-neighbour implementation.


In [5]:
from privacy_utility_framework.metrics.privacy import PrivacyMetricManager
from privacy_utility_framework.metrics.privacy.distance import (
    DCRCalculator, NNDRCalculator, AdversarialAccuracyCalculator,
)

def privacy_eval(syn, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    p = PrivacyMetricManager()
    p.add_metric([
        DCRCalculator(real, syn, original_name="real_min",
                      synthetic_name=name, preprocess=True),
        NNDRCalculator(real, syn, original_name="real_min",
                       synthetic_name=name, preprocess=True),
        AdversarialAccuracyCalculator(real, syn, original_name="real_min",
                                      synthetic_name=name, preprocess=True,
                                      backend="auto"),
    ])
    results = p.evaluate_all()
    row = {"model": name}
    for key, value in results.items():
        print(f"  {key}: {value}")
        row[str(key)] = value
    return row

priv_rows = [privacy_eval(syn_ctgan, "CTGAN"),
             privacy_eval(syn_ctab, "CTAB-GAN+")]



CTGAN
  DCRCalculator('real_min', 'CTGAN'): 0.3674156163680818
  NNDRCalculator('real_min', 'CTGAN'): 0.8636653373227271
  AdversarialAccuracyCalculator('real_min', 'CTGAN'): 0.7967711348621467

CTAB-GAN+
  DCRCalculator('real_min', 'CTAB-GAN+'): 0.2161000412584071
  NNDRCalculator('real_min', 'CTAB-GAN+'): 0.8384896026295017
  AdversarialAccuracyCalculator('real_min', 'CTAB-GAN+'): 0.6080668082904126


## 5. Similarity and utility metrics

Uses `UtilityMetricManager` with the similarity/utility calculators from
`examples/utility.py`: KS similarity, correlation, mutual information,
Jensen-Shannon similarity, and basic statistics. Wasserstein is computed
separately below because its calculator takes evaluation-time arguments.


In [6]:
from privacy_utility_framework.metrics.utility.utility_metric_manager import (
    UtilityMetricManager,
)
from privacy_utility_framework.metrics.utility.statistical import (
    BasicStatsCalculator, CorrelationCalculator, JSCalculator,
    KSCalculator, MICalculator,
)

def utility_eval(syn, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    p = UtilityMetricManager()
    p.add_metric([
        KSCalculator(real, syn, original_name="real_min", synthetic_name=name),
        CorrelationCalculator(real, syn, original_name="real_min", synthetic_name=name),
        MICalculator(real, syn, original_name="real_min", synthetic_name=name),
        JSCalculator(real, syn, original_name="real_min", synthetic_name=name),
        BasicStatsCalculator(real, syn, original_name="real_min", synthetic_name=name),
    ])
    results = p.evaluate_all()
    row = {"model": name}
    for key, value in results.items():
        print(f"  {key}: {value}")
        row[str(key)] = value
    return row

util_rows = [utility_eval(syn_ctgan, "CTGAN"),
             utility_eval(syn_ctab, "CTAB-GAN+")]



CTGAN
Method CorrelationMethod.PEARSON was used.
  KSCalculator('real_min', 'CTGAN'): 0.8513057260484431
  CorrelationCalculator('real_min', 'CTGAN'): nan
  MICalculator('real_min', 'CTGAN'): 0.9602351520855519
  JSCalculator('real_min', 'CTGAN'): 0.8296006602925914
  BasicStatsCalculator('real_min', 'CTGAN'): {'mean': 0.029239950519644394, 'median': 0.044632820779466675, 'var': 0.010064093861197157}

CTAB-GAN+
Method CorrelationMethod.PEARSON was used.
  KSCalculator('real_min', 'CTAB-GAN+'): 0.8344776950260405
  CorrelationCalculator('real_min', 'CTAB-GAN+'): nan
  MICalculator('real_min', 'CTAB-GAN+'): 0.9548985193067914
  JSCalculator('real_min', 'CTAB-GAN+'): 0.8437467540517444
  BasicStatsCalculator('real_min', 'CTAB-GAN+'): {'mean': 0.021014219158924635, 'median': 0.03265936649555415, 'var': 0.00483178882029503}


## 6. Wasserstein distance

The Wasserstein calculator takes its method and sample size at evaluation time,
so it is run on its own. A sampled Wasserstein estimate is used for speed:
200 records per sample, averaged over 10 samples, with a fixed seed so that
the value is reproducible.


In [7]:
from privacy_utility_framework.metrics.utility.statistical import (
    WassersteinCalculator, WassersteinMethod,
)

def wasserstein_eval(syn, name):
    calc = WassersteinCalculator(real, syn)
    np.random.seed(RANDOM_STATE)   # FEST samples with np.random: fix it for reproducibility
    res = calc.evaluate(metric=WassersteinMethod.WASSERSTEIN_SAMPLE,
                        n_samples=200, n_iterations=10)
    print(f"{name}: Wasserstein (sample) = {res:.4f}")
    return {"model": name, "Wasserstein": float(res)}

ws_rows = [wasserstein_eval(syn_ctgan, "CTGAN"),
           wasserstein_eval(syn_ctab, "CTAB-GAN+")]


Sampled Wasserstein Distance: 0.6354668639378416,                     with n_samples=200, n_iterations=10
CTGAN: Wasserstein (sample) = 0.6355
Sampled Wasserstein Distance: 0.4722424453131334,                     with n_samples=200, n_iterations=10
CTAB-GAN+: Wasserstein (sample) = 0.4722


## 7. Consolidate and save results

All blocks are merged per model and written to a single CSV, following the
naming convention of the other frameworks. Because FEST returns dictionaries
whose keys encode the metric and dataset names, the columns are taken as they
come and can be tidied later for the write-up.


In [8]:
from functools import reduce

frames = []
for rows in (priv_rows, util_rows, ws_rows):
    frames.append(pd.DataFrame(rows).set_index("model"))

fest_all = reduce(lambda a, b: a.join(b), frames).reset_index()
fest_all.to_csv(DATA_DIR / "fest_all_results.csv", index=False)
print("Saved: fest_all_results.csv\n")

with pd.option_context("display.max_columns", None, "display.width", 200):
    print(fest_all.to_string(index=False))


Saved: fest_all_results.csv

    model  DCRCalculator('real_min', 'CTGAN')  NNDRCalculator('real_min', 'CTGAN')  AdversarialAccuracyCalculator('real_min', 'CTGAN')  DCRCalculator('real_min', 'CTAB-GAN+')  NNDRCalculator('real_min', 'CTAB-GAN+')  AdversarialAccuracyCalculator('real_min', 'CTAB-GAN+')  KSCalculator('real_min', 'CTGAN')  CorrelationCalculator('real_min', 'CTGAN')  MICalculator('real_min', 'CTGAN')  JSCalculator('real_min', 'CTGAN')                                                   BasicStatsCalculator('real_min', 'CTGAN')  KSCalculator('real_min', 'CTAB-GAN+')  CorrelationCalculator('real_min', 'CTAB-GAN+')  MICalculator('real_min', 'CTAB-GAN+')  JSCalculator('real_min', 'CTAB-GAN+')                                             BasicStatsCalculator('real_min', 'CTAB-GAN+')  Wasserstein
    CTGAN                            0.367416                             0.863665                                            0.796771                                     NaN                

---
**Done.** If every cell ran, `fest_all_results.csv` holds the consolidated FEST
scores and `fest_all` is available in memory.

Interpretation notes for the write-up:
* DCR and NNDR here are FEST's own implementation and should be compared with the
  manual DCR/NNDR analysis (values may differ because of FEST's internal
  preprocessing and distance strategy, but the ordering between models is the
  informative part).
* NNAA (AdversarialAccuracy) is new: values near 0.5 suggest the synthetic data
  is hard to distinguish from real at the record level, while higher values
  indicate the adversary separates real from synthetic more easily.
* The similarity metrics (KS, correlation, MI, JS) are complements/similarities
  (higher = more similar), triangulating with SynthEval, Synthcity and SDMetrics.
* Wasserstein is a distance: lower = more similar.


In [9]:
# Diagnose the NaN in the correlation metric
from privacy_utility_framework.metrics.utility.statistical import CorrelationCalculator, CorrelationMethod

# try Spearman instead of Pearson
calc = CorrelationCalculator(real, syn_ctgan, original_name="real_min", synthetic_name="CTGAN")
import inspect
print(inspect.signature(calc.evaluate))

(method=<CorrelationMethod.PEARSON: 'pearson'>)


In [10]:
# Tidy reshape of the results (one column per metric, one row per model)
def tidy_privacy(rows):
    out = []
    for r in rows:
        model = r["model"]
        clean = {"model": model}
        for k, v in r.items():
            if k == "model": 
                continue
            # Extract the metric name before the "("
            metric = str(k).split("(")[0].replace("Calculator", "")
            clean[metric] = v
        out.append(clean)
    return out

import pandas as pd
priv_tidy = pd.DataFrame(tidy_privacy(priv_rows)).set_index("model")
util_tidy = pd.DataFrame(tidy_privacy(util_rows)).set_index("model")
ws_tidy   = pd.DataFrame(ws_rows).set_index("model")

fest_clean = priv_tidy.join(util_tidy).join(ws_tidy).reset_index()
print(fest_clean.to_string(index=False))
fest_clean.to_csv(DATA_DIR / "fest_all_results_clean.csv", index=False)
print("\nSaved: fest_all_results_clean.csv")

    model      DCR     NNDR  AdversarialAccuracy       KS  Correlation       MI       JS                                                                                  BasicStats  Wasserstein
    CTGAN 0.367416 0.863665             0.796771 0.851306          NaN 0.960235 0.829601 {'mean': 0.029239950519644394, 'median': 0.044632820779466675, 'var': 0.010064093861197157}     0.635467
CTAB-GAN+ 0.216100 0.838490             0.608067 0.834478          NaN 0.954899 0.843747   {'mean': 0.021014219158924635, 'median': 0.03265936649555415, 'var': 0.00483178882029503}     0.472242

Saved: fest_all_results_clean.csv


In [11]:
# Try Spearman correlation (more robust than Pearson for mixed/ordinal data)
from privacy_utility_framework.metrics.utility.statistical import CorrelationMethod

for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]:
    calc = CorrelationCalculator(real, syn, original_name="real_min", synthetic_name=name)
    res = calc.evaluate(method=CorrelationMethod.SPEARMAN)
    print(f"{name}: Correlation (Spearman) = {res}")
    

Method CorrelationMethod.SPEARMAN was used.
CTGAN: Correlation (Spearman) = nan
Method CorrelationMethod.SPEARMAN was used.
CTAB-GAN+: Correlation (Spearman) = nan


In [12]:
# Why is the correlation NaN? Inspect what FEST computes internally.
import numpy as np

# 1. Are there zero-variance columns? (a constant column breaks the correlation)
print("Zero-variance columns in real data:")
print([c for c in real.columns if real[c].nunique() <= 1])
print("Zero-variance columns in CTGAN data:")
print([c for c in syn_ctgan.columns if syn_ctgan[c].nunique() <= 1])

# 2. Are there hidden NaNs in the data?
print("\nNaNs in real:", real.isna().sum().sum())
print("NaNs in CTGAN:", syn_ctgan.isna().sum().sum())

Zero-variance columns in real data:
['def_pay']
Zero-variance columns in CTGAN data:
['def_pay']

NaNs in real: 0
NaNs in CTGAN: 0
